# 2. Model training

Training a ResNet34 classifier on the topoplots produced in notebook 1.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ocular import manifest, metrics, splits
from ocular.train import TrainConfig, train

MANIFEST = Path("../data/manifest.csv")
ARTIFACTS = Path("../artifacts")

frame = manifest.read(MANIFEST)
print(manifest.summarise(frame))

## Splitting by recording

Segments from one recording are highly correlated. They share a participant, an electrode
montage and a session, and one recording contributes hundreds of segments.

A random split over segments therefore places the same participant in both training and
validation, and the resulting score measures recognition of known participants rather than
generalisation to new ones.

In [ ]:
shuffled = frame.sample(frac=1.0, random_state=0)
cut = int(len(shuffled) * 0.8)
naive_train, naive_val = shuffled.iloc[:cut], shuffled.iloc[cut:]

shared = set(naive_train["group"]) & set(naive_val["group"])
print(f"random segment split: {len(shared)} of {frame['group'].nunique()} "
      f"recordings appear in both halves")

Splits are drawn over recordings instead, so no participant appears in more than one split.

In [ ]:
split = splits.make(frame, seed=91)
assigned = split.assign(frame)

print(splits.summarise(assigned))
print()
print("recordings in more than one split:",
      int((assigned.groupby("group")["split"].nunique() > 1).sum()))

Recordings are assigned by hashing their name rather than by shuffling, so adding a
recording later does not reassign the existing ones. The chosen split is stored in the
model checkpoint, so evaluation uses the same held out set.

In [ ]:
for name in splits.SPLITS:
    print(f"{name:6} {len(split.of(name)):>3} recordings")

## Class distribution

Blinks are one of four event types, and resting data yields more segments than the others
because it is cut with a sliding window. The classes are therefore uneven.

Rather than discarding data, the minority class is oversampled during training with a
weighted sampler.

In [ ]:
train_frame = assigned[assigned["split"] == "train"]
print(train_frame["event"].value_counts())
print()
print(train_frame["label"].value_counts(normalize=True).round(3))

## Training

A ResNet34 pretrained on ImageNet, with a new two class head. The dataset contains tens of
thousands of images but only a few dozen participants, so transfer learning is used rather
than training from scratch.

Training runs in two phases. The head is trained first with the backbone frozen, so the
randomly initialised layer does not propagate large gradients into the pretrained weights.
The whole network is then fine tuned at a lower learning rate.

Augmentation is limited to small crops and rotations. Horizontal flips are excluded because
mirroring the scalp converts a leftward saccade into a rightward one. Colour jitter is
excluded because the colormap encodes signal polarity.

The command line equivalent is `ocular train`.

In [ ]:
results = train(
    MANIFEST,
    ARTIFACTS,
    TrainConfig(
        architecture="resnet34",
        head_epochs=3,
        finetune_epochs=8,
        batch_size=32,
        seed=91,
    ),
)

In [ ]:
history = pd.DataFrame(results["history"])
history[["phase", "epoch", "train_loss", "val_loss",
         "train_balanced_accuracy", "val_balanced_accuracy"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("loss")
axes[1].plot(history["train_balanced_accuracy"], label="train")
axes[1].plot(history["val_balanced_accuracy"], label="val")
axes[1].set_title("balanced accuracy")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.show()

## Validation results

Balanced accuracy is reported alongside accuracy because the classes are uneven.

Validation participants are held out from training, so these figures estimate performance
on an unseen recording.

In [ ]:
print(metrics.format_report("validation", results["validation"]))

Notebook 3 evaluates the model on the test recordings and compares it against ICA.